<a href="https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
!pip install -q duckdb datasets huggingface_hub   #connecting the dataset- install the required libraries

In [3]:
# Read the HF_TOKEN from colab
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if HF_TOKEN else "Token not found")

Token loaded successfully!


In [4]:
#connect DuckDB to Hugging face
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

print("Connected successfully!")

Connected successfully!


In [5]:
DATASET = "hf://datasets/FlyRank/internship-warehouse" # the dataset path

In [6]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [7]:
feature_df = con.sql(f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 1000
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
numeric_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position"
]

feature_df[numeric_cols] = feature_df[numeric_cols].fillna(0)

boolean_cols = [
    "client_has_gsc",
    "client_has_ga4"
]

feature_df[boolean_cols] = feature_df[boolean_cols].fillna(False)

feature_df.head()

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,False
1,2026-03-01,content_05597932fe4da067,1,0,0,True,False
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,False
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,False
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,False


In [9]:
feature_df.columns

Index(['report_date', 'content_hash_id', 'gsc_impressions', 'gsc_clicks',
       'gsc_sum_position', 'client_has_gsc', 'client_has_ga4'],
      dtype='object')

### Build the Feature Vector

I built a feature vector using the March 2026 warehouse data. The selected features include Google Search Console metrics (`gsc_impressions`, `gsc_clicks`, and `gsc_sum_position`) along with the client connection status (`client_has_gsc` and `client_has_ga4`). Missing numeric values are filled with 0, and missing boolean values are filled with `False` to create a clean dataset for later analysis. The `report_date` and `content_hash_id` columns are kept for reference and tracking but are not intended to be used as predictive features.

In [10]:
import pandas as pd

feature_df = con.sql(f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 1000
""").df()

# Fill missing numeric values
numeric_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position"
]

feature_df[numeric_cols] = feature_df[numeric_cols].fillna(0)

# Fill missing boolean values
boolean_cols = [
    "client_has_gsc",
    "client_has_ga4"
]

feature_df[boolean_cols] = feature_df[boolean_cols].fillna(False)

feature_df.head()

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,False
1,2026-03-01,content_05597932fe4da067,1,0,0,True,False
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,False
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,False
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,False


In [11]:
feature_df.isnull().sum()

,0
report_date,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_sum_position,0
client_has_gsc,0
client_has_ga4,0


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes

| Feature | Meaning | Missing Values | Categorical? | Available Before Prediction? |
|----------|---------|----------------|--------------|------------------------------|
| gsc_impressions | Number of times a webpage appeared in Google Search results | No missing values | No | Yes |
| gsc_clicks | Number of clicks received from Google Search | No missing values | No | Yes, but it will be checked for possible data leakage because CTR depends on clicks |
| gsc_sum_position | Sum of search result positions for the page | No missing values | No | Yes |
| client_has_gsc | Indicates whether the client has Google Search Console connected | No missing values | Boolean | Yes |
| client_has_ga4 | Indicates whether the client has Google Analytics 4 connected | No missing values | Boolean | Yes |

In [13]:
feature_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   report_date       1000 non-null   datetime64[us]
 1   content_hash_id   1000 non-null   object        
 2   gsc_impressions   1000 non-null   int64         
 3   gsc_clicks        1000 non-null   int64         
 4   gsc_sum_position  1000 non-null   int64         
 5   client_has_gsc    1000 non-null   bool          
 6   client_has_ga4    1000 non-null   bool          
dtypes: bool(2), datetime64[us](1), int64(3), object(1)
memory usage: 41.1+ KB


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage Hunt- ANSWER

I reviewed the selected features for possible data leakage.

- `gsc_clicks` is a potential leakage feature because CTR is calculated using clicks and impressions. If CTR is the prediction target, using `gsc_clicks` as an input would indirectly provide information about the target.
- The remaining features (`gsc_impressions`, `gsc_sum_position`, `client_has_gsc`, and `client_has_ga4`) are available before prediction and are not directly derived from the target.
- No future-window features or client-identifying information were included in the feature vector.

In [15]:
# Check the selected features
print(feature_df.columns)

# Potential leakage feature
print("\nPotential leakage feature:")
print("gsc_clicks")

print("\nReason:")
print("CTR = gsc_clicks / gsc_impressions")
print("If CTR is the prediction target, gsc_clicks contains information used to calculate the target.")

Index(['report_date', 'content_hash_id', 'gsc_impressions', 'gsc_clicks',
       'gsc_sum_position', 'client_has_gsc', 'client_has_ga4'],
      dtype='object')

Potential leakage feature:
gsc_clicks

Reason:
CTR = gsc_clicks / gsc_impressions
If CTR is the prediction target, gsc_clicks contains information used to calculate the target.


In [19]:
# Inspect the feature vector
feature_df.info()

print("\nSelected features:")
print(feature_df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   report_date       1000 non-null   datetime64[us]
 1   content_hash_id   1000 non-null   object        
 2   gsc_impressions   1000 non-null   int64         
 3   gsc_clicks        1000 non-null   int64         
 4   gsc_sum_position  1000 non-null   int64         
 5   client_has_gsc    1000 non-null   bool          
 6   client_has_ga4    1000 non-null   bool          
dtypes: bool(2), datetime64[us](1), int64(3), object(1)
memory usage: 41.1+ KB

Selected features:
['report_date', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'client_has_gsc', 'client_has_ga4']


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### What I Excluded and Why

- `content_hash_id` – Excluded because it is only an identifier and does not contain predictive information.
- `report_date` – Excluded because it is used only for filtering the time window and is not intended as a predictive feature.
- `gsc_clicks` – Excluded from the final model because it is used to calculate CTR and may cause data leakage if CTR is the prediction target.

In [20]:
final_features = feature_df.drop(
    columns=["content_hash_id", "report_date", "gsc_clicks"]
)

final_features.head()

,gsc_impressions,gsc_sum_position,client_has_gsc,client_has_ga4
0,20,67,True,False
1,1,0,True,False
2,125,616,True,False
3,7,28,True,False
4,11,25,True,False


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.